# Early dropout sur CIFAR-10 : état des lieux critique
### MLP, CNN et ResNet compact, d'après Liu et al. (2023), « Dropout Reduces Underfitting »

Article : [PMLR 202, 2023](https://proceedings.mlr.press/v202/liu23aq.html) · [code officiel](https://github.com/facebookresearch/dropout)

## Objectif

L'article affirme qu'un dropout appliqué **seulement au début de l'entraînement** (« early dropout ») aide les modèles en **sous-apprentissage**. Il le montre sur ImageNet-1K avec des transformers (et quelques variantes), et il dit peu de choses sur ce que « début » veut dire : 50 époques par défaut, et un effet observé de 5 à 300 époques sur 600.

Ce notebook **ne reproduit pas l'article**. Il teste ses affirmations à petite échelle, sur CIFAR-10, avec trois architectures vues en cours (MLP, CNN, ResNet) et **le même protocole partout**.

## Les deux questions auxquelles on veut répondre

1. **L'early dropout marche-t-il à notre échelle, et sur quels modèles ?**
2. **Existe-t-il une durée de dropout optimale selon le type de modèle ?** Que veut dire « early » : les 10 % premières époques ? 30 % ? 50 % ?

## Démarche (pour chaque architecture)

| Étape | Ce que l'on fait | Pourquoi |
|---|---|---|
| **1. Régime** | Balayage largeur du modèle × durée d'entraînement, avec et sans dropout classique | L'article ne prédit un gain que si le modèle est en sous-apprentissage, c'est-à-dire s'il est meilleur **sans** dropout. Il faut donc trouver (et pas supposer) ce régime |
| **2. Comparaison** | Dans le régime trouvé : sans dropout, dropout classique, early dropout à 10 %, 30 %, 50 % du budget, plus un contrôle late dropout. 5 graines | Répondre aux questions 1 et 2 |
| **3. Test** | Score sur le test officiel, une seule fois, pour toutes les stratégies | Le test ne sert jamais à choisir quoi que ce soit |

En plus, on mesure les **métriques de gradient de l'article** (norme du gradient, distance au point initial, variance et erreur de direction) pour tester le *mécanisme* qu'il propose, pas seulement le résultat final.

## Plan

0. Lire l'article : résultats, métriques, différences avec notre protocole, différences appariées
1. Configuration et stratégies de dropout
2. Données
3. Architectures (MLP, CNN, ResNet avec BatchNorm)
4. Diagnostics de gradient
5. Entraînement
6. Vérifications automatiques
7. Outils statistiques
8. **Étape 1** : balayage et choix du régime
9. **Étape 2** : comparaison des stratégies, durée de la phase de dropout, diagnostics
10. **Étape 3** : test final
11. Discussion et limites

## Mode d'emploi

Les quatre calculs lourds sont derrière des interrupteurs (`RUN_SWEEP`, `RUN_MAIN`, `RUN_TEST`, plus `QUICK_RUN` pour une version miniature). Ils sont tous à `False` au départ. Les cellules d'analyse **relisent les résultats sauvegardés sur disque** : on peut donc redémarrer le noyau sans rien perdre. Conseil : lancer d'abord tout le notebook avec `QUICK_RUN = True` pour vérifier que la chaîne complète tourne, puis refaire avec `QUICK_RUN = False`.

## 0. Ce que dit l'article

### Idée centrale

- Le dropout est classiquement un outil contre le **sur-apprentissage** : il augmente la perte d'entraînement et réduit l'écart train/test.
- L'article montre qu'au **début** de l'entraînement, le dropout a un autre effet : il rend les gradients de mini-lots plus **cohérents entre eux** et plus **alignés** avec le gradient du jeu d'entraînement entier. Cela contrebalance le bruit de la descente de gradient stochastique.
- **Early dropout** : on applique le dropout avant une certaine époque, puis on le coupe. Selon l'article, cela abaisse la perte d'entraînement **finale** des modèles en sous-apprentissage, par rapport à l'absence de dropout *et* au dropout classique.
- **Late dropout** : l'idée symétrique pour les modèles en sur-apprentissage (pas de dropout au début, dropout à la fin). On l'utilise ici comme **contrôle** (même durée que l'early dropout, mais à la fin).

### Comment l'article définit le sous-apprentissage

Critère purement pratique (section 4) : si un modèle **généralise mieux avec un dropout classique**, il est en régime de sur-apprentissage ; s'il **performe mieux sans dropout**, il est en régime de sous-apprentissage. Ce régime dépend de l'architecture, des données **et des autres paramètres d'entraînement**. C'est exactement ce critère que l'étape 1 de notre notebook applique.

### Les deux hyperparamètres

1. **La durée de la phase de dropout.** L'article dit que le choix est robuste de 1 % à 50 % des époques totales. Par défaut : 50 époques. Dans la figure 9, l'effet est visible de 5 à 300 époques sur 600.
2. **Le taux de dropout p**, « moyennement robuste » : l'article note (figure 10) que l'effet de l'early dropout dépend fortement du taux sur ViT-T.

### Résultats principaux (ImageNet-1K, petits modèles de 5 à 20 millions de paramètres)

- Sur ViT-T, un dropout classique à 0,1 fait passer l'accuracy de 73,9 % à 67,9 %. L'early dropout la fait passer à 74,3 % (tableau 2).
- Les gains de l'early dropout sont **petits** : de l'ordre de 0,2 à 0,9 point selon les modèles, pour des écarts-types d'environ 0,1 à 0,2 point sur 3 graines.
- Sur ViT-B (plus gros, donc en sur-apprentissage), l'early dropout n'apporte rien (tableau 3c). L'effet est donc lié au régime.
- L'effet diminue quand la taille de lot augmente (figure 12), ce qui correspond à l'hypothèse « le dropout réduit l'erreur de gradient due au mini-lot ».
- Observation surprenante (annexe G) : pendant la phase de dropout, la perte d'entraînement n'est **pas** plus basse. Elle baisse nettement **après** la coupure. L'article suggère que l'early dropout ne va pas plus vite mais atteint peut-être un meilleur optimum.

### Calendrier du dropout

Par défaut l'article décroît linéairement le taux de p à 0 pendant la phase. Un taux **constant** puis coupé marche aussi (tableau 3b, annexe D), avec une phase plus courte (20 époques au lieu de 50) : l'annexe D explique que c'est la « dose cumulée » de dropout qui compte. **Nous utilisons le calendrier constant**, le plus simple à décrire.

### Les métriques introduites par l'article (section 3)

L'article compare deux entraînements de ViT-T (sans dropout / dropout à 0,1) et introduit quatre mesures. Toutes sont calculées sur des **gradients de mini-lots** g (un gradient par lot, par rapport à tous les paramètres).

| Métrique | Définition | Ce que l'article observe | Ce que cela implique |
|---|---|---|---|
| **Norme du gradient** ‖g‖₂ | Norme L2 du gradient du modèle entier (même si des neurones sont masqués) | Plus petite avec dropout | Le modèle avec dropout fait des **pas plus petits** |
| **Distance du modèle** ‖θ_t − θ_0‖₂ | Distance L2 entre les poids courants et les poids initiaux | Plus grande avec dropout | Malgré des pas plus petits, il s'éloigne **plus** de l'initialisation : il avance dans une direction plus **cohérente** (image des deux marcheurs : petites foulées régulières contre grandes foulées erratiques) |
| **GDV**, variance de direction du gradient | Moyenne, sur toutes les paires (i, j) de lots distincts, de la distance cosinus ½ × (1 − cos(g_i, g_j)) | Plus basse avec dropout jusqu'à environ 1000 itérations | Les gradients de lots différents **s'accordent mieux** entre eux |
| **GDE**, erreur de direction du gradient | Moyenne sur les lots de ½ × (1 − cos(g_lot, ĝ)), où ĝ est le gradient du **jeu d'entraînement entier**, calculé **sans masque** | Plus basse avec dropout au début, **plus haute** après environ 1000 itérations | Au début, le gradient bruité du mini-lot est plus proche de la « bonne » direction. Le point de bascule serait le moment où le dropout passe de « réduit le sous-apprentissage » à « réduit le sur-apprentissage » |

**Lecture biais-variance (article).** Sans dropout, le gradient d'un mini-lot est un estimateur non biaisé du gradient du jeu entier. Avec dropout, chaque lot passe dans un sous-réseau différent : l'estimateur devient **biaisé**, mais sa **variance baisse fortement**, ce qui réduit l'erreur globale. Cela empêche le modèle de se faire « balloter » par des lots individuels pendant la phase où il change le plus.

### Ce que notre notebook en fait

Cette explication produit des prédictions que l'on peut vérifier :

- **P1** : l'early dropout abaisse la perte d'entraînement finale des modèles en sous-apprentissage (c'est la mesure **principale** de notre étape 2).
- **P2** : pendant la phase de dropout, GDV et GDE sont plus bas avec masques qu'avec le réseau complet.
- **P3** : la distance au point initial est plus grande avec early dropout.

On les mesure en section 4 et dans l'étape 2. Deux adaptations à connaître : (a) on estime ĝ sur 4096 images du train et non sur tout le jeu ; (b) on mesure les gradients sur des images propres (sans augmentation ni mixup), alors que l'entraînement utilise de l'augmentation. Ce sont des limites, pas des détails. Repère utile : chez nous, une époque fait 352 itérations, donc les « environ 1000 itérations » de l'article correspondent à environ 3 époques.

### Nos choix pour reproduire l'article à notre échelle

| Aspect | Article | Ce notebook | Pourquoi / limite |
|---|---|---|---|
| Données | ImageNet-1K, 1,2 M images | CIFAR-10 : 45 000 train, 5 000 validation, 10 000 test | Jeu vu en cours, calcul accessible |
| Modèles | ViT-T, Mixer-S, ConvNeXt-F, Swin-F (5 à 20 M paramètres) | MLP, CNN, ResNet compact **avec BatchNorm**, largeur variable | Architectures du cours. Un écart entre architectures concerne leur configuration complète (capacité, normalisation...), pas seulement leur « type » |
| Optimiseur | AdamW, lr 4e-3, cosinus, warmup 50 époques, 300 ou 600 époques | AdamW, lr 1e-3, cosinus, warmup 1 époque, budget de 10, 20 ou 40 époques selon le balayage | Budget réduit |
| Taille de lot | 4096 (environ 310 itérations par époque) | 128 (352 itérations par époque) | Nombre d'itérations par époque comparable. L'article dit que l'effet baisse avec le lot ; le nôtre est petit, ce qui est plutôt favorable à l'early dropout |
| Régularisation | Recette ConvNeXt : RandAugment, mixup, cutmix, random erasing, label smoothing, weight decay 0,05 | Crop + flip + cutout + mixup 0,4, **weight decay 0** | On garde de l'augmentation pour entretenir un sous-apprentissage. Cela **fabrique en partie** le régime : on discute ce point en fin de notebook |
| Position du dropout | Plusieurs couches dans le réseau | **Une seule couche**, juste avant la couche de classification | Même point fonctionnel pour les trois modèles, mais bien moins de masques que dans l'article |
| Taux de dropout | p choisi parmi 0,1 / 0,2 / 0,3 par modèle | p = 0,3 unique | Limite : l'article dit que le résultat dépend du taux |
| Calendrier | Linéaire décroissant (défaut), constant en annexe D | Constant puis coupé | L'annexe D montre que les deux marchent |
| Durée de la phase | 50 époques par défaut, 5 à 300 sur 600 | **10 %, 30 %, 50 %** du budget d'entraînement | Plage 1 % à 50 % de l'article. Durée relative au budget de chaque architecture |
| Régime | « Meilleur sans dropout » | Même critère, mesuré par balayage largeur × durée **sur la validation** | Utiliser la validation garde le test intact pour la fin |
| Graines | 3 | 5 (3 pour le balayage) | Intervalles plus étroits |
| Mesure principale | Perte d'entraînement finale et accuracy | Perte d'entraînement propre (moyenne des 5 dernières époques), puis accuracy de validation, puis test | Même logique que l'article |

**Pourquoi des graines différentes pour le balayage et pour l'étape 2 ?** Si l'on choisit le régime là où l'effet du dropout classique est le plus négatif **sur certaines graines**, puis qu'on réévalue sur ces mêmes graines, on surestime cet effet (on a choisi le meilleur « hasard »). On utilise donc des graines **neuves** pour l'étape 2, qui sert alors aussi de confirmation du régime.

**Pourquoi BatchNorm dans le ResNet ?** C'est la normalisation vue en cours. En mode entraînement elle utilise les statistiques du lot ; en inférence, des statistiques glissantes. Le dropout étant placé **après** toutes les BatchNorm, il ne perturbe pas leurs statistiques.

### Différences appariées : qu'est-ce que c'est et pourquoi ?

**Problème.** Deux entraînements avec des graines différentes ne donnent pas le même résultat : initialisation, ordre des lots, augmentations. Cette variabilité (de l'ordre de quelques dixièmes de point d'accuracy ; l'article rapporte lui-même des écarts-types de 0,1 à 0,2 point sur ImageNet) est **du même ordre que les effets qu'on cherche** (l'article rapporte des gains de 0,2 à 0,9 point, stochastic depth inclus). Comparer des moyennes de groupes séparés noierait l'effet dans le bruit.

**Solution : l'appariement par graine.** Dans ce notebook, pour une graine donnée, **toutes les stratégies partagent** :

- les mêmes poids initiaux ;
- le même ordre des lots, les mêmes augmentations et les mêmes tirages de mixup (générateur aléatoire dédié aux données) ;

et ne diffèrent que par **les masques de dropout** (autre générateur). Pour chaque graine i, on calcule alors la **différence** d = mesure(stratégie A, graine i) − mesure(stratégie B, graine i). Le bruit propre à la graine s'annule en grande partie dans d.

**Statistique.** On obtient n différences d_1, ..., d_n (n = 5). On en prend la moyenne et un **intervalle de confiance à 95 % de Student** : moyenne ± t × s / √n, où s est l'écart-type des d_i et t le quantile de Student à n − 1 degrés de liberté (t = 2,78 pour n = 5, mais 4,30 pour n = 3 : avec 3 graines, l'intervalle est large).

**Verdict.** Si l'intervalle est **entièrement du même côté de 0**, l'effet est détecté (amélioration ou dégradation selon la métrique). S'il contient 0, on écrit « indétectable ». **Indétectable ne veut pas dire nul** : avec 5 graines la puissance est limitée, seuls les effets assez grands ressortent.

**Précautions de lecture.**

- Les intervalles sont **ponctuels** : on ne corrige pas pour les nombreuses comparaisons. Plus on regarde de comparaisons, plus on risque de croiser un faux positif.
- Student suppose des différences à peu près gaussiennes ; avec 5 valeurs, on ne peut pas vraiment le vérifier.
- On rapporte aussi `n_neg`, le nombre de graines où la différence est négative, pour voir si l'effet est **régulier** ou tiré par une seule graine.
- Une seule comparaison est **prévue d'avance** comme principale : early_30 contre none sur la perte d'entraînement. Toutes les autres sont exploratoires.

### Interrupteurs d'exécution

Tout est désactivé par défaut. L'ordre à suivre : `QUICK_RUN` pour vérifier, puis `RUN_SWEEP` (étape 1), puis `RUN_MAIN` (étape 2), et `RUN_TEST` **en dernier**, une seule fois.

In [1]:
# Dépendances (à décommenter si besoin) :
# %pip install torch torchvision numpy pandas scipy matplotlib tqdm

import os
# Requis par les algorithmes déterministes de PyTorch sur GPU (à définir avant toute initialisation CUDA).
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import math, random, time, json, gc, hashlib
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import torch
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from IPython.display import display

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device :', device)

# Reproductibilité : on demande des algorithmes déterministes (simple avertissement si impossible).
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

PyTorch 2.15.0+cu132 | device : cuda


In [2]:
# ======================= INTERRUPTEURS =======================
# Tout est désactivé par défaut : lire d'abord les cellules Markdown, puis activer dans l'ordre.
QUICK_RUN = False    # True : mini-version (jeu réduit, très peu d'époques) pour vérifier que tout tourne
RUN_SWEEP = True    # Étape 1 : balayage largeur x durée -> régime de sous-apprentissage
RUN_MAIN = True     # Étape 2 : comparaison des stratégies de dropout dans le régime retenu
RUN_TEST = True     # Étape 3 : score final sur le test (à lancer UNE seule fois, tout à la fin)
FORCE_RERUN = False  # True : ré-entraîne même si un résultat identique est déjà sauvegardé

## 1. Configuration commune et stratégies de dropout

Un **seul** objet `Config` regroupe les hyperparamètres communs à toutes les architectures et toutes les stratégies. Seules deux valeurs changent d'une configuration à l'autre : `width_mult` (la taille du modèle) et `epochs` (la durée). Ce sont justement les deux variables que l'étape 1 fait varier pour trouver le régime de sous-apprentissage.

**Les grilles de l'étape 1** (modifiables ci-dessous) :

- largeur × 0,5 / × 1 / × 2 : un modèle plus petit sous-apprend davantage ;
- durée de 10 / 20 / 40 époques : un entraînement plus court sous-apprend davantage.

La partie la plus coûteuse du notebook est le balayage. Si c'est trop long, réduire `WIDTH_GRID`, `EPOCH_GRID` ou `SWEEP_SEEDS`.

In [3]:
@dataclass(frozen=True)
class Config:
    '''Hyperparamètres COMMUNS à toutes les architectures et à toutes les stratégies.'''
    # --- Optimisation ---
    batch_size: int = 128
    lr: float = 1e-3
    lr_min_ratio: float = 0.01      # plancher de la décroissance cosinus (1 % du lr max)
    warmup_epochs: float = 1.0      # montée linéaire du lr pendant 1 époque
    weight_decay: float = 0.0       # AdamW sans weight decay : aucune autre régularisation explicite
    dropout_p: float = 0.30         # taux de dropout, identique pour standard, early et late
    # --- Augmentation de données (entretient un régime de sous-apprentissage) ---
    aug_crop_pad: int = 4           # crop aléatoire après padding de 4 pixels (0 = désactivé)
    aug_flip: bool = True           # flip horizontal
    aug_cutout: int = 8             # carré effacé de 8x8 pixels (0 = désactivé)
    aug_mixup_alpha: float = 0.4    # mixup (0 = désactivé)
    # --- Données ---
    val_size: int = 5000
    train_subset: int = 0           # 0 = tout le train (45 000) ; >0 : sous-échantillon (QUICK_RUN)
    train_eval_size: int = 5000     # images du train utilisées pour mesurer la perte d'entraînement
    split_seed: int = 2026
    # --- Les deux variables du régime : fixées par le balayage (étape 1) ---
    width_mult: float = 1.0         # facteur appliqué à la largeur de chaque couche
    epochs: int = 30                # budget d'entraînement
    # --- Diagnostics de gradient (section 4) ---
    grad_batches: int = 32          # nombre de mini-lots de sondage
    grad_batch_size: int = 128
    grad_ref_size: int = 4096       # images servant à estimer le gradient du jeu complet
    grad_mask_seed: int = 1234


SWEEP_SEEDS = (0, 1, 2)                       # étape 1 : graines distinctes de celles de l'étape 2
MAIN_SEEDS = (42, 123, 456, 789, 1011)        # étape 2
ARCHITECTURES = ('mlp', 'cnn', 'resnet')
WIDTH_GRID = (0.5, 1.0, 2.0)                  # facteurs de largeur testés à l'étape 1
EPOCH_GRID = (10, 20, 40)                     # budgets testés à l'étape 1
EARLY_FRACTIONS = (0.10, 0.30, 0.50)          # durée de la phase de dropout, en fraction du budget
LATE_FRACTION = 0.30                          # durée du contrôle late dropout
DIAG_EPOCHS = (0, 1, 2, 3, 5, 8, 12, 20, 30, 40)   # époques où l'on mesure les gradients (0 = initialisation)
LOG_EVERY = 10
BASE = Config()

if QUICK_RUN:   # mini-version : 3000 images, quelques époques
    BASE = replace(BASE, train_subset=3000, train_eval_size=1000, grad_batches=8, grad_ref_size=1024)
    MAIN_SEEDS = (42, 123, 456)
    WIDTH_GRID, EPOCH_GRID = (0.5, 1.0), (3, 6)
    DIAG_EPOCHS = (0, 1, 2, 3, 6)

OUT = Path('./results_early_dropout_v2_quick' if QUICK_RUN else './results_early_dropout_v2')
OUT.mkdir(parents=True, exist_ok=True)
print('Dossier de résultats :', OUT.resolve())

Dossier de résultats : /home/nibir/projects/projet dl final/results_early_dropout_v2


### Les stratégies de dropout

Une stratégie est un **calendrier** : à chaque époque, quel taux de dropout utilise-t-on ?

| Nom | Calendrier | Rôle |
|---|---|---|
| `none` | p = 0 tout le long | Référence sans dropout |
| `standard` | p = 0,3 tout le long | Dropout classique (critère du régime) |
| `early_10`, `early_30`, `early_50` | p = 0,3 pendant les 10 % / 30 % / 50 % **premières** époques, puis 0 | Early dropout, avec trois définitions de « début » |
| `late_30` | p = 0 puis p = 0,3 pendant les 30 % **dernières** époques | Contrôle : même durée que `early_30`, mais à la fin |

Le nombre d'époques actives est `fraction × budget`, arrondi au plus proche (demi-entiers vers le haut), avec au moins une époque. Exemple pour 20 époques : `early_10` = 2 époques, `early_30` = 6, `early_50` = 10. Le nom `early_30` signifie donc **30 % du budget** (et pas 30 époques).

**Pourquoi le contrôle `late_30` ?** Si `early_30` et `late_30` donnent le même résultat, ce n'est peut-être pas le *moment* du dropout qui compte mais sa *quantité*. C'est une question critique sur l'article, qui défend explicitement le début de l'entraînement.

In [4]:
def round_half_up(x):
    return math.floor(x + 0.5)

PRIMARY = 'early_30'                                          # contraste principal : early_30 contre none
EARLY = [f'early_{round(100 * f)}' for f in EARLY_FRACTIONS]  # ['early_10', 'early_30', 'early_50']
LATE = f'late_{round(100 * LATE_FRACTION)}'                   # 'late_30'
STRATEGIES = ['none', 'standard'] + EARLY + [LATE]
assert PRIMARY in EARLY

def n_active_epochs(strategy, total):
    '''Nombre d'époques pendant lesquelles le dropout est actif.'''
    if strategy == 'none':
        return 0
    if strategy == 'standard':
        return total
    pct = int(strategy.split('_')[1])                          # early_30 -> 30 %
    return max(1, min(total, round_half_up(pct / 100 * total)))

def dropout_rate(strategy, epoch, total, p):
    '''Taux de dropout utilisé pendant l'époque `epoch` (numérotée de 1 à total).'''
    n = n_active_epochs(strategy, total)
    if strategy.startswith('late'):
        active = epoch > total - n          # fenêtre à la FIN de l'entraînement
    else:
        active = epoch <= n                 # none (n=0), standard (n=total), early : fenêtre au DÉBUT
    return p if active else 0.0

print('Stratégies :', STRATEGIES)

Stratégies : ['none', 'standard', 'early_10', 'early_30', 'early_50', 'late_30']


## 2. Données

- Le train officiel (50 000 images) est coupé par une permutation fixe en **45 000 images d'entraînement** et **5 000 de validation**. Le **test officiel** (10 000 images) est chargé à part et ne sert qu'à l'étape 3.
- Tout est stocké sur le GPU en `uint8` ; l'augmentation et la normalisation se font à la volée sur GPU (rapide).
- **Deux sources d'aléa séparées** : les masques de dropout utilisent le générateur par défaut de PyTorch, tandis que l'ordre des lots, les augmentations et le mixup utilisent un générateur dédié (`data_gen`). C'est ce qui permet l'appariement : pour une graine donnée, deux stratégies voient exactement les mêmes images dans le même ordre avec les mêmes augmentations.
- On mesure la perte d'entraînement sur un sous-ensemble fixe de 5 000 images du train, en **images propres** (sans augmentation) et en mode évaluation (dropout coupé). C'est la mesure « de l'ajustement » : elle ne dépend pas des masques ni des augmentations de l'époque.

In [5]:
from torchvision import datasets

def load_cifar10(root='./data'):
    '''Retourne ((X_train, y_train), (X_test, y_test)) en uint8 (N, 3, 32, 32) et int64.'''
    def to_tensors(d):
        return (torch.from_numpy(d.data).permute(0, 3, 1, 2).contiguous(),
                torch.tensor(d.targets, dtype=torch.long))
    return (to_tensors(datasets.CIFAR10(root, train=True, download=True)),
            to_tensors(datasets.CIFAR10(root, train=False, download=True)))

(X_all, y_all), (X_te, y_te) = load_cifar10()

# Découpage fixe train / validation (le test officiel reste à part)
perm = np.random.default_rng(BASE.split_seed).permutation(len(X_all))
val_ids, train_ids = perm[:BASE.val_size], perm[BASE.val_size:]
if BASE.train_subset:
    train_ids = train_ids[:BASE.train_subset]
as_idx = lambda a: torch.from_numpy(np.ascontiguousarray(a))
Xtr, ytr = X_all[as_idx(train_ids)].to(device), y_all[as_idx(train_ids)].to(device)
Xva, yva = X_all[as_idx(val_ids)].to(device), y_all[as_idx(val_ids)].to(device)
Xte, yte = X_te.to(device), y_te.to(device)

# Sous-ensemble fixe du train pour mesurer la perte d'entraînement propre
eval_ids = np.random.default_rng(BASE.split_seed + 1).choice(
    len(Xtr), size=min(BASE.train_eval_size, len(Xtr)), replace=False)
Xev, yev = Xtr[as_idx(eval_ids).to(device)], ytr[as_idx(eval_ids).to(device)]
del X_all, y_all, X_te, y_te
print('Train', len(Xtr), '| Validation', len(Xva), '| Train-eval', len(Xev), '| Test', len(Xte))

# Normalisation par canal (moyennes et écarts-types de CIFAR-10, reprises par la plupart des implémentations de référence)
MEAN = torch.tensor((0.4914, 0.4822, 0.4465), device=device).view(1, 3, 1, 1)
STD = torch.tensor((0.2470, 0.2435, 0.2616), device=device).view(1, 3, 1, 1)

def prep(x_u8):
    '''uint8 -> float32 normalisé (images PROPRES, utilisées pour toutes les mesures).'''
    return (x_u8.float().div(255) - MEAN) / STD

def augment(x_u8, gen, cfg):
    '''Augmentation sur GPU : crop aléatoire + flip + normalisation + cutout.
    Tout l'aléa vient du générateur `gen` (dédié aux données, indépendant du dropout).'''
    B = x_u8.shape[0]
    x = x_u8.float().div(255)
    pad = cfg.aug_crop_pad
    if pad > 0:
        # Crop aléatoire : on ajoute un cadre de zéros, puis on découpe 32x32 à une position aléatoire par image.
        xp = F.pad(x, (pad, pad, pad, pad))
        oy = torch.randint(0, 2 * pad + 1, (B,), device=device, generator=gen)
        ox = torch.randint(0, 2 * pad + 1, (B,), device=device, generator=gen)
        ar = torch.arange(32, device=device)
        b_idx = torch.arange(B, device=device)[:, None, None, None]
        c_idx = torch.arange(3, device=device)[None, :, None, None]
        rows = (oy[:, None] + ar)[:, None, :, None]
        cols = (ox[:, None] + ar)[:, None, None, :]
        x = xp[b_idx, c_idx, rows, cols]
    if cfg.aug_flip:
        flip = torch.rand(B, device=device, generator=gen) < 0.5      # une image sur deux est retournée
        x = torch.where(flip[:, None, None, None], x.flip(3), x)
    x = (x - MEAN) / STD
    if cfg.aug_cutout > 0:
        # Cutout : un carré de côté s est mis à 0 à une position aléatoire.
        s = cfg.aug_cutout
        cy = torch.randint(0, 32, (B,), device=device, generator=gen)
        cx = torch.randint(0, 32, (B,), device=device, generator=gen)
        ar = torch.arange(32, device=device)
        in_y = (ar[None] >= (cy[:, None] - s // 2)) & (ar[None] < (cy[:, None] + s - s // 2))
        in_x = (ar[None] >= (cx[:, None] - s // 2)) & (ar[None] < (cx[:, None] + s - s // 2))
        x = x.masked_fill((in_y[:, :, None] & in_x[:, None, :])[:, None], 0.0)
    return x

Train 45000 | Validation 5000 | Train-eval 5000 | Test 10000


## 3. Architectures

Les trois modèles ont la **même structure de haut niveau** : `features` (le corps du réseau) → `dropout` → `classifier` (une couche linéaire vers 10 classes). Le dropout est donc au **même point fonctionnel** dans les trois modèles : sur la représentation finale, juste avant la classification.

| Modèle | Corps (largeurs de base) | Détail |
|---|---|---|
| **MLP** | 3072 → 512 → 256 → 128 | Trois couches denses avec ReLU |
| **CNN** | convolutions 64, 128, 256, puis couche dense de 128 | Max-pooling après les deux premières convolutions, moyenne globale à la fin |
| **ResNet compact** | convolution 16, puis 3 blocs résiduels de 16, 32, 64 canaux, puis couche dense de 128 | **BatchNorm** après chaque convolution, raccourcis (projection 1×1 quand les dimensions changent) |

**`width_mult`** multiplie toutes les largeurs (× 0,5 = modèle deux fois plus étroit). C'est la variable « taille de modèle » du balayage. L'avant-dernière couche (celle qui précède le dropout) est mise à l'échelle aussi, donc le dropout s'applique à une représentation de largeur variable.

**Bloc résiduel.** La branche principale (conv → BN → ReLU → conv → BN) apprend une *correction* de l'entrée ; on additionne l'entrée (le « raccourci »), puis ReLU. Quand le nombre de canaux ou la résolution changent, le raccourci est une convolution 1×1 (avec BN) pour que les formes correspondent.

In [6]:
def scaled(n, mult):
    '''Largeur n multipliée par mult, au moins 4.'''
    return max(4, int(round(n * mult)))

class GlobalAvgPool(nn.Module):
    '''Moyenne sur les dimensions spatiales : (B, C, H, W) -> (B, C).'''
    def forward(self, x):
        return x.mean(dim=(2, 3))

class Net(nn.Module):
    '''Structure commune aux trois modèles : features -> dropout -> classifier.'''
    def __init__(self, features, feat_dim, p):
        super().__init__()
        self.features = features
        self.dropout = nn.Dropout(p)
        self.classifier = nn.Linear(feat_dim, 10)

    def forward(self, x):
        return self.classifier(self.dropout(self.features(x)))

class ResidualBlock(nn.Module):
    '''Bloc résiduel : sortie = ReLU( branche(x) + raccourci(x) ).'''
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        if c_in == c_out and stride == 1:
            self.shortcut = nn.Identity()                    # mêmes formes : on additionne directement
        else:
            self.shortcut = nn.Sequential(                   # formes différentes : projection 1x1
                nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
        return F.relu(out + self.shortcut(x))

def build_model(arch, cfg):
    '''Construit le modèle `arch` avec la largeur cfg.width_mult et le dropout cfg.dropout_p.'''
    m, p = cfg.width_mult, cfg.dropout_p
    if arch == 'mlp':
        h1, h2, h3 = (scaled(n, m) for n in (256, 128, 64))
        feats = nn.Sequential(
            nn.Flatten(), nn.Linear(3 * 32 * 32, h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.ReLU(),
            nn.Linear(h2, h3), nn.ReLU())
        return Net(feats, h3, p)
    if arch == 'cnn':
        c1, c2, c3, fc = (scaled(n, m) for n in (48, 96, 192, 96))
        feats = nn.Sequential(
            nn.Conv2d(3, c1, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(c2, c3, 3, padding=1), nn.ReLU(),
            GlobalAvgPool(), nn.Linear(c3, fc), nn.ReLU())
        return Net(feats, fc, p)
    if arch == 'resnet':
        c1, c2, c3, fc = (scaled(n, m) for n in (16, 32, 64, 128))
        feats = nn.Sequential(
            nn.Conv2d(3, c1, 3, padding=1, bias=False), nn.BatchNorm2d(c1), nn.ReLU(),
            ResidualBlock(c1, c1, 1),          # (B, c1, 32, 32)
            ResidualBlock(c1, c2, 2),          # (B, c2, 16, 16)
            ResidualBlock(c2, c3, 2),          # (B, c3, 8, 8)
            GlobalAvgPool(), nn.Linear(c3, fc), nn.ReLU())
        return Net(feats, fc, p)
    raise ValueError(arch)

def count_params(arch, cfg):
    return sum(p.numel() for p in build_model(arch, cfg).parameters())

## 4. Diagnostics de gradient (les métriques de l'article)

Cette fonction mesure, sur un modèle à un instant donné (un « checkpoint »), les métriques de la section 3 de l'article. Elle fonctionne dans **deux modes, avec exactement les mêmes poids** :

- `eval` : aucun masque. On mesure le gradient du réseau complet ;
- `masked` : dropout actif à p = 0,3, **quelle que soit la stratégie d'entraînement**. Les masques sont tirés avec une graine fixe, donc identiques d'une stratégie à l'autre.

**Ce qui est calculé** (m = 32 mini-lots de 128 images, tirés à part du gradient de référence) :

- `grad_norm` : norme moyenne des m gradients ;
- `gdv` : **variance de direction** = ½ × (1 − cosinus moyen entre les paires de lots distincts) ;
- `gde` : **erreur de direction** = moyenne de ½ × (1 − cosinus entre un gradient de lot et le gradient de référence ĝ). Comme dans l'article, ĝ est calculé **sans masque** ; nous l'estimons sur 4096 images du train au lieu du jeu entier.

**Trois comparaisons utiles** (on les calcule en section 9) :

1. *effet direct des masques* : `masked` contre `eval` sur les mêmes poids (stratégie `none`) ;
2. *comparaison de l'article* : modèle entraîné avec early dropout, mesuré avec masques, contre modèle sans dropout, mesuré sans masque ;
3. *effet de trajectoire* : modèle early contre modèle `none`, tous deux mesurés **sans** masque. Cela dit si le dropout a changé la position du modèle dans l'espace des poids.

**Limites.** Les mesures utilisent des images propres alors que l'entraînement utilise augmentation et mixup. Pour le ResNet, le mode `eval` utilise les statistiques glissantes de la BatchNorm.

In [7]:
def flat_grad(model, params, idx):
    '''Gradient de la perte (images propres) sur les images idx, aplati en un seul vecteur.'''
    loss = F.cross_entropy(model(prep(Xtr[idx])), ytr[idx])
    return torch.cat([g.reshape(-1) for g in torch.autograd.grad(loss, params)])

def mean_gradient(model, params, idx, chunk=1024):
    '''Gradient moyen sur beaucoup d'images, calculé par morceaux pour économiser la mémoire.'''
    total = 0
    for i in range(0, len(idx), chunk):
        part = idx[i:i + chunk]
        total = total + flat_grad(model, params, part) * len(part)
    return total / len(idx)

def make_probe_split(seed, cfg):
    '''Lots de sondage + images de référence, disjoints. Dépendent de la graine, pas de la stratégie.'''
    n_probe = cfg.grad_batches * cfg.grad_batch_size
    assert n_probe + cfg.grad_ref_size <= len(Xtr), 'jeu de sondage trop grand pour le jeu d entrainement'
    p = np.random.default_rng(777 + seed).permutation(len(Xtr))
    return as_idx(p[:n_probe]).to(device), as_idx(p[n_probe:n_probe + cfg.grad_ref_size]).to(device)

def gradient_diagnostics(model, probe_idx, ref_idx, epoch, cfg):
    '''Métriques de gradient de l'article (section 3) mesurées sur un checkpoint.

    Deux modes sur les MÊMES poids :
      - eval   : sans masque de dropout (gradient du réseau complet) ;
      - masked : dropout actif au taux cfg.dropout_p, quelle que soit la stratégie d'entraînement.
    Retourne une liste de deux dictionnaires (un par mode).'''
    params = list(model.parameters())
    bs = cfg.grad_batch_size
    m = len(probe_idx) // bs
    assert m >= 2, 'il faut au moins 2 lots de sondage'
    was_training, old_p = model.training, model.dropout.p
    model.eval()                                  # BatchNorm : statistiques d'inférence
    devices = [torch.cuda.current_device()] if device.type == 'cuda' else []
    rows = []
    with torch.random.fork_rng(devices=devices):  # le diagnostic ne perturbe pas l'aléa de l'entraînement
        # Gradient de référence g_hat : réseau complet (sans masque), moyenné sur ref_idx.
        model.dropout.train(False)
        g_hat = mean_gradient(model, params, ref_idx)
        g_hat = g_hat / g_hat.norm().clamp_min(1e-12)
        for mode in ('eval', 'masked'):
            model.dropout.train(mode == 'masked')         # masques actifs seulement en mode masked
            model.dropout.p = cfg.dropout_p
            torch.manual_seed(cfg.grad_mask_seed + epoch)  # mêmes masques pour toutes les stratégies
            G = torch.stack([flat_grad(model, params, probe_idx[b * bs:(b + 1) * bs]) for b in range(m)])
            norms = G.norm(dim=1)
            U = G / norms.clamp_min(1e-12)[:, None]       # directions unitaires, matrice m x D
            mean_cos = ((U @ U.T).sum() - m) / (m * (m - 1))   # cosinus moyen entre paires DISTINCTES
            rows.append(dict(epoch=epoch, mode=mode,
                             grad_norm=norms.mean().item(),
                             gdv=(0.5 * (1 - mean_cos)).item(),          # variance de direction
                             gde=(0.5 * (1 - U @ g_hat)).mean().item())) # erreur de direction
    model.dropout.p = old_p
    model.train(was_training)
    return rows

## 5. Entraînement

`train_one` entraîne **un** modèle (une architecture, une stratégie, une graine) et sauvegarde l'historique (JSON) et les poids finaux (`.pt`). Si le même entraînement existe déjà sur disque, il est relu au lieu d'être refait : on peut donc interrompre et relancer.

**Ce qui est identique pour toutes les stratégies d'une même graine :** initialisation des poids, ordre des lots, augmentations, tirages de mixup, schedule du learning rate. **Ce qui diffère :** uniquement le taux de dropout à chaque époque.

**Schedule du learning rate.** Montée linéaire pendant 1 époque, puis décroissance cosinus vers 1 % du lr maximal. Comme la décroissance dépend du budget total, un modèle à 20 époques et un à 40 époques n'ont pas le même schedule : c'est pourquoi chaque budget a ses **propres** baselines.

**Mixup.** On mélange chaque lot avec une version permutée de lui-même (poids λ tiré d'une loi bêta) et on pondère la perte de la même façon. C'est une régularisation forte qui entretient le sous-apprentissage.

**Mesures de fin d'époque** : perte et accuracy sur 5 000 images propres du train, perte et accuracy sur la validation, distance du modèle à son initialisation ‖θ_t − θ_0‖₂ (métrique de l'article). Les diagnostics de gradient sont faits à quelques époques seulement (`DIAG_EPOCHS`) et seulement pour `none` et `early_30`, pour limiter le coût.

In [8]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()

def flat_params(model):
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])

@torch.no_grad()
def evaluate(model, X, y, bs=2500):
    '''Perte et accuracy (fraction) en mode eval (dropout coupé), sur images propres.'''
    was_training = model.training
    model.eval()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    for i in range(0, len(X), bs):
        logits = model(prep(X[i:i + bs]))
        yy = y[i:i + bs]
        loss_sum += F.cross_entropy(logits, yy, reduction='sum')
        correct += (logits.argmax(1) == yy).sum()
    model.train(was_training)
    return {'loss': loss_sum.item() / len(X), 'accuracy': correct.item() / len(X)}

def lr_factor(step, total, warmup):
    '''Facteur multiplicatif du lr : montée linéaire, puis cosinus jusqu'à lr_min_ratio.'''
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return BASE.lr_min_ratio + (1 - BASE.lr_min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))

def mixup_loss(model, x, y, gen, mix_rng, cfg):
    '''Perte avec mixup : on mélange le lot avec une version permutée de lui-même.'''
    if cfg.aug_mixup_alpha <= 0:
        return F.cross_entropy(model(x), y)
    lam = float(mix_rng.beta(cfg.aug_mixup_alpha, cfg.aug_mixup_alpha))
    j = torch.randperm(len(x), generator=gen, device=device)
    logits = model(lam * x + (1 - lam) * x[j])
    return lam * F.cross_entropy(logits, y) + (1 - lam) * F.cross_entropy(logits, y[j])

def run_signature(arch, strategy, seed, cfg, diagnose):
    '''Empreinte du run : si elle change, le résultat sauvegardé n'est plus réutilisé.'''
    return json.dumps({'arch': arch, 'strategy': strategy, 'seed': seed, 'cfg': asdict(cfg),
                       'diagnose': diagnose, 'version': 3}, sort_keys=True)

def train_one(arch, strategy, seed, cfg, folder, diagnose=False):
    '''Entraîne un modèle avec un calendrier de dropout donné, puis sauvegarde l'historique (JSON)
    et les poids finaux (.pt). Un résultat identique déjà sauvegardé est relu au lieu d'être recalculé.'''
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    stem = f'{arch}_w{cfg.width_mult}_e{cfg.epochs}_{strategy}_s{seed}'
    json_path, ckpt_path = folder / f'{stem}.json', folder / f'{stem}.pt'
    sig = run_signature(arch, strategy, seed, cfg, diagnose)
    if json_path.exists() and ckpt_path.exists() and not FORCE_RERUN:
        saved = json.loads(json_path.read_text(encoding='utf-8'))
        if saved['signature'] == sig:
            saved['checkpoint'] = str(ckpt_path)
            return saved

    # --- 1. Initialisation : la graine fixe les poids ET les masques de dropout ---
    seed_everything(seed)
    model = build_model(arch, cfg).to(device)
    params = list(model.parameters())
    theta0 = flat_params(model).clone()           # poids initiaux, pour la distance du modèle
    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    n = len(Xtr)
    steps_per_epoch = math.ceil(n / cfg.batch_size)
    total_steps = cfg.epochs * steps_per_epoch
    warmup_steps = cfg.warmup_epochs * steps_per_epoch
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: lr_factor(s, total_steps, warmup_steps))

    # --- 2. Aléa des DONNÉES : générateur séparé de celui des masques de dropout ---
    data_gen = torch.Generator(device=device)
    data_gen.manual_seed(seed + 10_000)           # ordre des lots, augmentations, mixup
    mix_rng = np.random.default_rng(seed + 20_000)

    # --- 3. Diagnostics de gradient (optionnels) : à l'initialisation puis à quelques époques ---
    diag_epochs = [e for e in DIAG_EPOCHS if e <= cfg.epochs] if diagnose else []
    if diagnose:
        probe_idx, ref_idx = make_probe_split(seed, cfg)
    grads = gradient_diagnostics(model, probe_idx, ref_idx, 0, cfg) if 0 in diag_epochs else []

    # --- 4. Boucle d'entraînement ---
    history, t_train = [], 0.0
    for epoch in range(1, cfg.epochs + 1):
        sync()
        t0 = time.perf_counter()
        model.train()
        model.dropout.p = dropout_rate(strategy, epoch, cfg.epochs, cfg.dropout_p)   # calendrier du dropout
        order = torch.randperm(n, generator=data_gen, device=device)
        for i in range(0, n, cfg.batch_size):
            idx = order[i:i + cfg.batch_size]
            x, y = augment(Xtr[idx], data_gen, cfg), ytr[idx]
            loss = mixup_loss(model, x, y, data_gen, mix_rng, cfg)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
        sync()
        t_train += time.perf_counter() - t0

        # Mesures de fin d'époque : images PROPRES, dropout coupé (evaluate passe en model.eval()).
        tr, va = evaluate(model, Xev, yev), evaluate(model, Xva, yva)
        rec = dict(epoch=epoch, lr=opt.param_groups[0]['lr'], dropout_p=model.dropout.p,
                   model_distance=(flat_params(model) - theta0).norm().item(),
                   train_loss=tr['loss'], train_acc=100 * tr['accuracy'],
                   val_loss=va['loss'], val_acc=100 * va['accuracy'])
        history.append(rec)
        if epoch in diag_epochs:
            grads += gradient_diagnostics(model, probe_idx, ref_idx, epoch, cfg)
        if epoch == 1 or epoch % LOG_EVERY == 0 or epoch == cfg.epochs:
            tl, vl, vacc, p_now = rec['train_loss'], rec['val_loss'], rec['val_acc'], rec['dropout_p']
            tqdm.write(f'{arch:6s} | {strategy:9s} | seed {seed} | ep {epoch:02d}/{cfg.epochs} | p={p_now:.1f} '
                       f'| train {tl:.3f} | val {vl:.3f} | val acc {vacc:.1f}')

    # --- 5. Sauvegarde ---
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, ckpt_path)
    out = dict(arch=arch, strategy=strategy, seed=seed, cfg=asdict(cfg), history=history,
               gradients=grads, train_seconds=t_train, signature=sig)
    json_path.write_text(json.dumps(out), encoding='utf-8')
    del model, opt
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    out['checkpoint'] = str(ckpt_path)
    return out

## 6. Vérifications automatiques

Ces tests rapides vérifient les briques sensibles avant de lancer les calculs longs : le calendrier de dropout (le bon nombre d'époques actives, au bon endroit), l'augmentation (déterministe pour un générateur donné, et qui modifie bien les images), la forme des sorties des modèles, et les diagnostics de gradient (valeurs dans [0, 1], état aléatoire préservé, masques qui changent bien le gradient). Si un `assert` échoue, ne pas continuer.

In [9]:
# --- 1. Calendrier de dropout ---
assert [dropout_rate('early_30', e, 20, 0.3) for e in (1, 6, 7, 20)] == [0.3, 0.3, 0.0, 0.0]
assert [dropout_rate('late_30', e, 20, 0.3) for e in (1, 14, 15, 20)] == [0.0, 0.0, 0.3, 0.3]
assert all(dropout_rate('standard', e, 20, 0.3) == 0.3 for e in range(1, 21))
assert all(dropout_rate('none', e, 20, 0.3) == 0.0 for e in range(1, 21))
assert n_active_epochs('early_10', 10) == 1 and n_active_epochs('early_10', 20) == 2
for E in EPOCH_GRID:
    print(f'{E:3d} époques ->', {s: n_active_epochs(s, E) for s in STRATEGIES}, '(époques avec dropout)')

# --- 2. Augmentation : déterministe pour un générateur donné, et elle modifie les images ---
g1 = torch.Generator(device=device).manual_seed(0)
g2 = torch.Generator(device=device).manual_seed(0)
a, b = augment(Xtr[:64], g1, BASE), augment(Xtr[:64], g2, BASE)
assert torch.equal(a, b) and a.shape == (64, 3, 32, 32)
assert not torch.allclose(a, prep(Xtr[:64]))

# --- 3. Modèles : formes de sortie et nombre de paramètres ---
for arch in ARCHITECTURES:
    for w in (0.5, 1.0, 2.0):
        m = build_model(arch, replace(BASE, width_mult=w))
        assert m(torch.randn(4, 3, 32, 32)).shape == (4, 10)
        print(f'{arch:6s} largeur x{w}: {count_params(arch, replace(BASE, width_mult=w)):>9,} paramètres')

# --- 4. Diagnostics de gradient ---
seed_everything(0)
m = build_model('mlp', BASE).to(device)
cfg_t = replace(BASE, grad_batches=4, grad_batch_size=64, grad_ref_size=512)
probe, ref = make_probe_split(0, cfg_t)
cpu_state = torch.get_rng_state()
gpu_state = torch.cuda.get_rng_state() if device.type == 'cuda' else None
rows = gradient_diagnostics(m, probe, ref, 1, cfg_t)
assert torch.equal(cpu_state, torch.get_rng_state())                         # l'aléa n'est pas perturbé
assert gpu_state is None or torch.equal(gpu_state, torch.cuda.get_rng_state())
r = {x['mode']: x for x in rows}
assert 0 <= r['eval']['gdv'] <= 1 and 0 <= r['eval']['gde'] <= 1             # distances cosinus normalisées
assert r['eval']['grad_norm'] != r['masked']['grad_norm']                    # les masques changent le gradient
del m
print('Vérifications : OK')

 10 époques -> {'none': 0, 'standard': 10, 'early_10': 1, 'early_30': 3, 'early_50': 5, 'late_30': 3} (époques avec dropout)
 20 époques -> {'none': 0, 'standard': 20, 'early_10': 2, 'early_30': 6, 'early_50': 10, 'late_30': 6} (époques avec dropout)
 40 époques -> {'none': 0, 'standard': 40, 'early_10': 4, 'early_30': 12, 'early_50': 20, 'late_30': 12} (époques avec dropout)
mlp    largeur x0.5:   404,010 paramètres
mlp    largeur x1.0:   828,490 paramètres
mlp    largeur x2.0: 1,738,890 paramètres
cnn    largeur x0.5:    57,802 paramètres
cnn    largeur x1.0:   228,490 paramètres
cnn    largeur x2.0:   908,554 paramètres
resnet largeur x0.5:    22,386 paramètres
resnet largeur x1.0:    87,002 paramètres
resnet largeur x2.0:   342,954 paramètres
Vérifications : OK


## 7. Outils statistiques et lecture des résultats

Cette cellule rassemble les outils utilisés partout ensuite :

- `t_ci` : moyenne et intervalle de confiance à 95 % de Student ;
- `paired_diff` : **la différence appariée** décrite plus haut (A − B, graine par graine, puis moyenne et intervalle) ;
- `verdict` : traduit l'intervalle en « amélioration », « dégradation » ou « indétectable », en tenant compte du sens de la métrique (perte : plus petit est mieux ; accuracy : plus grand est mieux) ;
- `summarize` : résume chaque entraînement par la **moyenne des 5 dernières époques** (moins sensible au bruit d'une seule époque que la valeur finale) ;
- `COMPARISONS` : la liste des comparaisons faites à l'étape 2.

**Les comparaisons de l'étape 2.**

- chaque stratégie contre `none` (effet de chacune) ;
- chaque early dropout contre `standard` (early fait-il mieux que le dropout classique ?) ;
- les durées entre elles, `early_30 − early_10`, `early_50 − early_10`, `early_50 − early_30` (question 2) ;
- `early_30 − late_30` (début contre fin, à durée égale).

In [10]:
def t_ci(x):
    '''Moyenne et intervalle de confiance à 95 % de Student (sur n valeurs).'''
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan
    if n < 2:
        return float(x.mean()), np.nan, np.nan
    half = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / math.sqrt(n)
    return float(x.mean()), float(x.mean() - half), float(x.mean() + half)

def paired_diff(df, metric, a, b, cond='strategy', by='seed'):
    '''Différence appariée a - b de `metric` pour chaque graine, puis moyenne et IC95 % de Student.

    df   : une ligne par (graine, condition)
    cond : colonne qui contient les conditions comparées (par défaut la stratégie)
    by   : colonne d'appariement (la graine)
    n_neg : nombre de graines où la différence est négative.'''
    piv = df.pivot_table(index=by, columns=cond, values=metric)
    if a not in piv.columns or b not in piv.columns:
        return dict(mean=np.nan, lo=np.nan, hi=np.nan, n=0, n_neg=0)
    d = (piv[a] - piv[b]).dropna().to_numpy()
    mean, lo, hi = t_ci(d)
    return dict(mean=mean, lo=lo, hi=hi, n=len(d), n_neg=int((d < 0).sum()))

def verdict(r, higher_is_better):
    '''Traduit l'intervalle de confiance en verdict, selon le sens de la métrique.'''
    if r['n'] < 3 or np.isnan(r['lo']):
        return 'n insuffisant'
    if r['lo'] > 0:                                   # intervalle entièrement au-dessus de 0
        return 'amélioration' if higher_is_better else 'dégradation'
    if r['hi'] < 0:                                   # intervalle entièrement en dessous de 0
        return 'dégradation' if higher_is_better else 'amélioration'
    return 'indétectable'                             # l'intervalle contient 0

def load_runs(folder):
    '''Relit tous les entraînements sauvegardés dans un dossier (marche après un redémarrage).'''
    runs = []
    for path in sorted(Path(folder).glob('*.json')):
        r = json.loads(path.read_text(encoding='utf-8'))
        r['checkpoint'] = str(path.with_suffix('.pt'))
        runs.append(r)
    return runs

def summarize(runs):
    '''Une ligne par entraînement : mesures moyennées sur les 5 dernières époques.'''
    rows = []
    for r in runs:
        h = pd.DataFrame(r['history'])
        last = h.tail(5)
        rows.append(dict(
            arch=r['arch'], strategy=r['strategy'], seed=r['seed'],
            width_mult=r['cfg']['width_mult'], epochs=r['cfg']['epochs'],
            train_loss=last.train_loss.mean(), val_loss=last.val_loss.mean(),
            train_acc=last.train_acc.mean(), val_acc=last.val_acc.mean(),
            gap=(last.val_loss.mean() - last.train_loss.mean()) / last.val_loss.mean()))   # écart relatif train/val
    return pd.DataFrame(rows)

# Comparaisons de l'étape 2 (A, B) = « A moins B »
COMPARISONS = [(s, 'none') for s in STRATEGIES if s != 'none']                        # effet de chaque stratégie
COMPARISONS += [(s, 'standard') for s in EARLY + [LATE]]                               # contre le dropout classique
COMPARISONS += [(EARLY[j], EARLY[i]) for j in range(len(EARLY)) for i in range(j)]     # durées entre elles
COMPARISONS += [(PRIMARY, LATE)]                                                       # début contre fin

def comparison_table(df, metrics):
    '''Toutes les comparaisons appariées, pour chaque architecture. metrics : liste de (colonne, plus_grand_est_mieux).'''
    rows = []
    for arch, g in df.groupby('arch'):
        for a, b in COMPARISONS:
            for col_name, higher in metrics:
                r = paired_diff(g, col_name, a, b)
                rows.append(dict(arch=arch, comparaison=f'{a} - {b}', metrique=col_name,
                                 **r, verdict=verdict(r, higher)))
    return pd.DataFrame(rows)

# Régime retenu à l'étape 1, éventuellement forcé à la main (ex. {'cnn': dict(width_mult=0.5, epochs=20)})
REGIME_OVERRIDE = {}

def load_regime():
    path = OUT / 'regime.json'
    regime = json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    regime.update(REGIME_OVERRIDE)
    return regime

def keep_regime(runs, regime):
    '''Ne garde que les entraînements qui correspondent au régime retenu pour leur architecture.'''
    return [r for r in runs if r['arch'] in regime
            and r['cfg']['width_mult'] == regime[r['arch']]['width_mult']
            and r['cfg']['epochs'] == regime[r['arch']]['epochs']]

# Couleurs et graphiques
COLORS = {'none': '#555555', 'standard': '#E69F00', 'early_10': '#9ECAE1', 'early_30': '#0072B2',
          'early_50': '#08306B', LATE: '#009E73'}
col = lambda s: COLORS.get(s, '#CC79A7')

def curve_ci(df, value, group):
    '''Moyenne et IC95 % de `value` pour chaque (groupe, époque), calculés sur les graines.'''
    rows = []
    for (g_name, epoch), g in df.groupby([group, 'epoch']):
        mean, lo, hi = t_ci(g[value].to_numpy())
        rows.append(dict(group=g_name, epoch=epoch, mean=mean, lo=lo, hi=hi))
    return pd.DataFrame(rows)

def delta_curve(h, value, base='none'):
    '''Différence appariée par graine (stratégie - base), époque par époque.'''
    piv = h.pivot_table(index=['seed', 'epoch'], columns='strategy', values=value)
    parts = []
    for s in piv.columns:
        if s != base:
            d = (piv[s] - piv[base]).rename('delta').reset_index()
            d['strategy'] = s
            parts.append(d)
    return pd.concat(parts)

def plot_band(ax, curves, groups, style=None):
    '''Trace la moyenne et la bande d'IC95 % pour chaque groupe.'''
    for g_name in groups:
        c = curves[curves.group == g_name].sort_values('epoch')
        color, ls = style[g_name] if style else (col(g_name), '-')
        ax.plot(c.epoch, c['mean'], color=color, ls=ls, label=g_name)
        ax.fill_between(c.epoch, c.lo, c.hi, color=color, alpha=0.12)

def finish(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / f'{name}.png', dpi=150)
    plt.show()

# Étape 1 : trouver le régime de sous-apprentissage

### Ce que l'on fait

Pour chaque architecture, on entraîne avec **`none`** et **`standard`** (dropout classique, p = 0,3) sur une grille **largeur × durée** (par défaut 3 × 3), avec **3 graines**. Pour chaque case, on calcule la différence appariée

> Δ = accuracy de validation(`standard`) − accuracy de validation(`none`)

C'est le critère de l'article : si le dropout classique **dégrade** l'accuracy, la case est un régime de **sous-apprentissage**.

### Règles de décision (fixées avant de voir les résultats)

1. **Case confirmée** : l'intervalle à 95 % de Δ est entièrement négatif. Le critère est strict, car avec 3 graines (t = 4,30) l'intervalle est large : seuls les effets nets sont retenus.
2. **Choix parmi les cases confirmées** : celle où `none` atteint la **meilleure accuracy de validation**. On cherche le modèle le plus performant qui reste en sous-apprentissage, pour éviter de tester l'early dropout sur un modèle minuscule et sans intérêt.
3. **Aucune case confirmée** : on retient la case où la moyenne de Δ est la plus négative et on l'**étiquette `confirmed = False`**. L'étape 2 s'exécute quand même, mais ses résultats sont à lire comme *hors du périmètre de l'article* : celui-ci ne prédit aucun gain hors sous-apprentissage. Cela reste un résultat valable.

### Pourquoi la validation et pas le test ?

La consigne parle d'accuracy de test. Mais choisir une configuration d'après le test le contaminerait : le score final ne serait plus une mesure honnête. Le critère est le même, appliqué à un jeu mis de côté pour ça.

### Coût

La cellule suivante affiche le nombre d'entraînements et d'époques avant de lancer quoi que ce soit. Chaque budget a ses propres baselines, car le schedule de learning rate dépend du budget.

In [ ]:
def sweep_jobs():
    '''Toutes les combinaisons (architecture, largeur, durée, stratégie, graine) du balayage.'''
    jobs = []
    for arch in ARCHITECTURES:
        for w in WIDTH_GRID:
            for e in EPOCH_GRID:
                cfg = replace(BASE, width_mult=w, epochs=e)
                for strategy in ('none', 'standard'):
                    for seed in SWEEP_SEEDS:
                        jobs.append((arch, strategy, seed, cfg))
    return jobs

jobs = sweep_jobs()
print(len(jobs), 'entraînements,', sum(j[3].epochs for j in jobs), 'époques au total')

if RUN_SWEEP:
    for arch, strategy, seed, cfg in tqdm(jobs):
        train_one(arch, strategy, seed, cfg, OUT / 'sweep_runs')
else:
    print('Balayage non lancé (RUN_SWEEP = False).')

162 entraînements, 3780 époques au total


  0%|          | 0/162 [00:00<?, ?it/s]

mlp    | none      | seed 0 | ep 01/10 | p=0.0 | train 1.790 | val 1.797 | val acc 36.6
mlp    | none      | seed 0 | ep 10/10 | p=0.0 | train 1.449 | val 1.471 | val acc 48.9
mlp    | none      | seed 1 | ep 01/10 | p=0.0 | train 1.780 | val 1.781 | val acc 36.9
mlp    | none      | seed 1 | ep 10/10 | p=0.0 | train 1.466 | val 1.490 | val acc 48.0
mlp    | none      | seed 2 | ep 01/10 | p=0.0 | train 1.755 | val 1.764 | val acc 37.2
mlp    | none      | seed 2 | ep 10/10 | p=0.0 | train 1.459 | val 1.487 | val acc 47.0
mlp    | standard  | seed 0 | ep 01/10 | p=0.3 | train 1.838 | val 1.841 | val acc 34.6
mlp    | standard  | seed 0 | ep 10/10 | p=0.3 | train 1.489 | val 1.525 | val acc 46.7
mlp    | standard  | seed 1 | ep 01/10 | p=0.3 | train 1.824 | val 1.831 | val acc 35.3
mlp    | standard  | seed 1 | ep 10/10 | p=0.3 | train 1.513 | val 1.537 | val acc 46.2
mlp    | standard  | seed 2 | ep 01/10 | p=0.3 | train 1.799 | val 1.796 | val acc 36.1
mlp    | standard  | seed 2 | ep

### Lecture du balayage

La table donne, pour chaque case (architecture, largeur, durée) :

- `params` : taille du modèle ;
- `val_acc_none` : accuracy de validation sans dropout ;
- `delta_acc`, `ic_bas`, `ic_haut` : Δ (en points d'accuracy) et son intervalle à 95 % ;
- `underfit` : `True` si l'intervalle est entièrement négatif (case confirmée) ;
- `train_acc_none` et `gap_none` : accuracy d'entraînement et écart relatif train/validation de la baseline. Ce sont des **indicateurs descriptifs** (un modèle en sous-apprentissage a plutôt une accuracy d'entraînement modeste et un petit écart), ils ne conditionnent pas le choix.

**Carte** : chaque case affiche Δ. Rouge = le dropout classique **dégrade** (sous-apprentissage), bleu = il **aide** (sur-apprentissage). Une étoile `*` marque les cases confirmées.

**Ce que l'article prédit** : le rouge devrait se concentrer sur les petits modèles et les entraînements courts, et virer au bleu vers les grands modèles et les longs entraînements. Si la carte ne ressemble pas à cela, c'est un résultat intéressant en soi. Le régime choisi est écrit dans `regime.json`, dont l'étape 2 se sert.

In [ ]:
def sweep_table(runs):
    '''Une ligne par case (architecture, largeur, durée) : effet du dropout classique sur la validation.'''
    s = summarize(runs)
    rows = []
    for (arch, w, e), g in s.groupby(['arch', 'width_mult', 'epochs']):
        r = paired_diff(g, 'val_acc', 'standard', 'none')        # accuracy val : standard - none
        base = g[g.strategy == 'none']
        rows.append(dict(
            arch=arch, width_mult=w, epochs=e,
            params=count_params(arch, replace(BASE, width_mult=w, epochs=e)),
            val_acc_none=base.val_acc.mean(), train_acc_none=base.train_acc.mean(), gap_none=base.gap.mean(),
            delta_acc=r['mean'], ic_bas=r['lo'], ic_haut=r['hi'], n=r['n'],
            underfit=bool(r['hi'] < 0)))                          # intervalle entièrement négatif
    return pd.DataFrame(rows)

def select_regime(table):
    '''Choisit une configuration (largeur, durée) par architecture, selon les règles fixées plus haut.'''
    chosen = {}
    for arch, g in table.groupby('arch'):
        confirmed = g[g.underfit]
        if len(confirmed):
            best, ok = confirmed.sort_values('val_acc_none', ascending=False).iloc[0], True
        else:
            best, ok = g.sort_values('delta_acc').iloc[0], False     # repli : Δ le plus négatif
        chosen[arch] = dict(width_mult=float(best.width_mult), epochs=int(best.epochs),
                            confirmed=ok, delta_acc=float(best.delta_acc))
    return chosen

def plot_sweep(table):
    fig, axes = plt.subplots(1, len(ARCHITECTURES), figsize=(5.2 * len(ARCHITECTURES), 4))
    lim = max(0.5, float(table.delta_acc.abs().max()))
    for ax, arch in zip(axes, ARCHITECTURES):
        t = table[table.arch == arch]
        delta = t.pivot(index='width_mult', columns='epochs', values='delta_acc')
        flag = t.pivot(index='width_mult', columns='epochs', values='underfit')
        im = ax.imshow(delta.values, cmap='RdBu', vmin=-lim, vmax=lim, origin='lower', aspect='auto')
        ax.set_xticks(range(len(delta.columns)))
        ax.set_xticklabels(delta.columns)
        ax.set_yticks(range(len(delta.index)))
        ax.set_yticklabels(delta.index)
        for i in range(delta.shape[0]):
            for j in range(delta.shape[1]):
                star = ' *' if flag.values[i, j] else ''
                ax.text(j, i, f'{delta.values[i, j]:+.1f}{star}', ha='center', va='center')
        ax.set_title(arch.upper())
        ax.set_xlabel('époques')
        ax.set_ylabel('facteur de largeur')
    fig.colorbar(im, ax=list(axes), label='standard - none : accuracy val (points)')
    fig.suptitle('Effet du dropout classique (rouge = dégrade = sous-apprentissage, * = confirmé)')
    fig.savefig(OUT / 'sweep_regime.png', dpi=150, bbox_inches='tight')
    plt.show()

sweep_runs = load_runs(OUT / 'sweep_runs')
if sweep_runs:
    sweep_df = sweep_table(sweep_runs)
    display(sweep_df.round(3))
    sweep_df.to_csv(OUT / 'sweep_table.csv', index=False)
    plot_sweep(sweep_df)
    chosen = select_regime(sweep_df)
    (OUT / 'regime.json').write_text(json.dumps(chosen, indent=2), encoding='utf-8')
    print('Régime retenu par architecture (écrit dans regime.json) :')
    display(pd.DataFrame(chosen).T)
else:
    print('Aucun résultat de balayage : activer RUN_SWEEP puis relancer cette cellule.')

# Étape 2 : comparer les stratégies dans le régime retenu

Pour chaque architecture, on se place dans la configuration (largeur, durée) retenue à l'étape 1 et on entraîne les **six stratégies** (`none`, `standard`, `early_10`, `early_30`, `early_50`, `late_30`) avec **5 graines neuves**.

### Ce que l'on mesure

| Mesure | Rôle | Statut |
|---|---|---|
| **Perte d'entraînement propre** (moyenne des 5 dernières époques) | Mesure de **l'ajustement**. C'est ce que l'article annonce d'améliorer (« lower final training loss ») | **Principale** : contraste prévu d'avance `early_30 − none` |
| Accuracy de validation | Transfert hors entraînement. L'article dit que l'accuracy s'améliore aussi | Secondaire |
| Perte de validation | Complément | Exploratoire |
| Diagnostics de gradient | Teste le **mécanisme** (P2, P3 plus haut) | Exploratoire |
| Test | Score final | Étape 3 |

### Ce que l'on doit retrouver si l'article tient

- Dans le régime confirmé, le dropout classique **dégrade** la validation (confirmation sur graines neuves).
- Pendant la fenêtre de dropout, la perte d'entraînement d'un early dropout est plutôt **au-dessus** de `none`. **Après la coupure**, elle redescend, et sa valeur finale est idéalement **en dessous** de `none` (annexe G, figure 13 de l'article).
- Un effet de **petite taille** (de l'ordre de dixièmes de point d'accuracy). Avec 5 graines, il ne sera détectable que si le bruit est faible.

Les diagnostics de gradient ne sont faits que pour `none` et `early_30` (coût).

In [ ]:
if RUN_MAIN:
    regime = load_regime()
    missing = [a for a in ARCHITECTURES if a not in regime]
    assert not missing, f'Régime manquant pour {missing} : lancer et analyser le balayage (étape 1).'
    jobs = []
    for arch in ARCHITECTURES:
        cfg = replace(BASE, width_mult=regime[arch]['width_mult'], epochs=regime[arch]['epochs'])
        for strategy in STRATEGIES:
            for seed in MAIN_SEEDS:
                jobs.append((arch, strategy, seed, cfg))
    print(len(jobs), 'entraînements,', sum(j[3].epochs for j in jobs), 'époques au total')
    for arch, strategy, seed, cfg in tqdm(jobs):
        # Diagnostics de gradient seulement pour la référence et le contraste principal.
        train_one(arch, strategy, seed, cfg, OUT / 'main_runs', diagnose=strategy in ('none', PRIMARY))
else:
    print('Étape 2 non lancée (RUN_MAIN = False).')

### 2a. Le régime est-il confirmé sur des graines neuves ?

Premier tableau : pour chaque architecture, la configuration retenue et le contraste `standard − none` sur l'accuracy de validation **avec les 5 graines neuves**. Si le dropout classique n'est plus clairement négatif ici, le choix de l'étape 1 était en partie dû au hasard (régression vers la moyenne) : il faut alors être très prudent sur la suite.

### 2b. Tableau des comparaisons appariées

Pour chaque architecture, chaque comparaison de la liste `COMPARISONS` et chaque métrique (perte d'entraînement, perte de validation, accuracy de validation) : différence moyenne, intervalle à 95 %, nombre de graines concernées, verdict.

Lecture : `early_30 − none` sur `train_loss` avec `verdict = amélioration` signifie que la perte d'entraînement finale baisse avec l'early dropout, de façon régulière (intervalle entièrement négatif). `n_neg` est le nombre de graines (sur 5) où la différence est négative. Toutes les lignes autres que le contraste principal sont **exploratoires**.

In [ ]:
regime = load_regime()
main_runs = keep_regime(load_runs(OUT / 'main_runs'), regime)

if main_runs:
    main = summarize(main_runs)
    main.to_csv(OUT / 'summary_main.csv', index=False)

    # 2a. Le régime est-il confirmé sur graines neuves ? (accuracy val : standard - none)
    rows = []
    for arch, g in main.groupby('arch'):
        r = paired_diff(g, 'val_acc', 'standard', 'none')
        base = g[g.strategy == 'none']
        cfg_a = replace(BASE, width_mult=float(g.width_mult.iloc[0]), epochs=int(g.epochs.iloc[0]))
        rows.append(dict(arch=arch, width_mult=cfg_a.width_mult, epochs=cfg_a.epochs, params=count_params(arch, cfg_a),
                         val_acc_none=base.val_acc.mean(), standard_moins_none=r['mean'], ic_bas=r['lo'], ic_haut=r['hi'],
                         verdict_acc_val=verdict(r, True), train_acc_none=base.train_acc.mean(), gap_none=base.gap.mean()))
    print('2a. Régime confirmé sur graines neuves ? (accuracy de validation, standard - none, en points)')
    display(pd.DataFrame(rows).round(3))

    # 2b. Comparaisons appariées
    metrics = [('train_loss', False), ('val_loss', False), ('val_acc', True)]
    cmp_df = comparison_table(main, metrics)
    cmp_df.to_csv(OUT / 'paired_comparisons.csv', index=False)
    print('Contraste PRINCIPAL (prévu d avance) : perte d entrainement, early_30 - none')
    display(cmp_df[(cmp_df.comparaison == f'{PRIMARY} - none') & (cmp_df.metrique == 'train_loss')].round(4))
    print('Toutes les comparaisons (exploratoires, IC ponctuels, sans correction de multiplicité) :')
    display(cmp_df.round(4))
else:
    print('Aucun résultat pour le régime retenu : activer RUN_MAIN (et faire d abord l étape 1).')

### 2c. Les courbes d'entraînement

Pour chaque architecture, six panneaux (moyenne sur les graines, bande = IC à 95 %) :

- **ligne du haut** : perte d'entraînement propre, perte de validation, accuracy de validation, toutes stratégies ;
- **ligne du bas** : les **différences appariées** à `none`, époque par époque (perte d'entraînement et accuracy de validation), puis la distance du modèle à son initialisation.

Les lignes verticales en pointillés marquent la **fin de la phase de dropout** de chaque early dropout.

**Ce que l'on regarde** :

1. Pendant la phase de dropout, la différence de perte d'entraînement est-elle positive (le dropout gêne) ? Que se passe-t-il **juste après** la coupure ? C'est le comportement de la figure 13 de l'article.
2. La bande de la différence finale contient-elle 0 ?
3. Que fait `standard` ? Il doit être nettement au-dessus de `none` (perte d'entraînement plus haute), c'est le coût du dropout permanent.
4. La **distance au point initial** (métrique de l'article) : avec early dropout, le modèle s'éloigne-t-il plus de son initialisation, comme l'article le rapporte pour ViT-T ?

In [ ]:
if main_runs:
    hist_df = pd.DataFrame([dict(arch=r['arch'], strategy=r['strategy'], seed=r['seed'], **rec)
                            for r in main_runs for rec in r['history']])
    hist_df.to_csv(OUT / 'history.csv', index=False)
    for arch in ARCHITECTURES:
        h = hist_df[hist_df.arch == arch]
        if h.empty:
            continue
        E = int(h.epoch.max())
        fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
        # (axe, métrique, titre, tracer la différence appariée à none ?)
        panels = [(axes[0, 0], 'train_loss', 'Perte train (images propres)', False),
                  (axes[0, 1], 'val_loss', 'Perte validation', False),
                  (axes[0, 2], 'val_acc', 'Accuracy validation (%)', False),
                  (axes[1, 0], 'train_loss', 'Différence de perte train vs none (appariée)', True),
                  (axes[1, 1], 'val_acc', 'Différence d accuracy val vs none (points, appariée)', True),
                  (axes[1, 2], 'model_distance', 'Distance du modèle à son initialisation', False)]
        for ax, value, title, as_delta in panels:
            if as_delta:
                curves = curve_ci(delta_curve(h, value), 'delta', 'strategy')
                groups = [s for s in STRATEGIES if s != 'none']
                ax.axhline(0, color='black', lw=0.8)
            else:
                curves, groups = curve_ci(h, value, 'strategy'), STRATEGIES
            plot_band(ax, curves, groups)
            ax.set_title(title)
            ax.set_xlabel('époque')
            ax.grid(alpha=0.3)
            for s in EARLY:                      # fin de la phase de dropout de chaque early dropout
                ax.axvline(n_active_epochs(s, E), color=col(s), ls=':', alpha=0.7)
        axes[0, 0].legend(fontsize=8)
        width = regime[arch]['width_mult']
        fig.suptitle(f'{arch.upper()} ({E} époques, largeur x{width}) : moyenne et IC95 % sur {h.seed.nunique()} graines')
        finish(fig, f'curves_{arch}')

### 2d. Question 2 : quelle durée de dropout ?

Ce graphique répond directement à la question « que veut dire *early* ? ». Pour chaque architecture :

- **axe horizontal** : la fraction du budget pendant laquelle le dropout est actif (10 %, 30 %, 50 % ; 100 % = dropout classique) ;
- **axe vertical** : la différence appariée avec `none`, avec son intervalle à 95 %. En haut, la **perte d'entraînement** (sous 0 = meilleur ajustement) ; en bas, l'**accuracy de validation** (au-dessus de 0 = mieux) ;
- le point vert est le contrôle `late_30` (même durée que `early_30`, mais à la fin).

**Comment conclure.** Une « durée optimale » n'existe que si les intervalles de deux durées sont **séparés**, ou si le tableau 2b (lignes `early_30 − early_10`, etc.) donne un verdict. Si les intervalles se chevauchent, la conclusion honnête est : « on ne sait pas départager ces durées avec 5 graines ». L'article lui-même affirme que la durée est peu critique entre 1 % et 50 % ; on teste si c'est vrai à notre échelle.

In [ ]:
def plot_duration(summary):
    fig, axes = plt.subplots(2, len(ARCHITECTURES), figsize=(15, 8))
    metrics = (('train_loss', 'Différence de perte train vs none'),
               ('val_acc', 'Différence d accuracy val vs none (points)'))
    for j, arch in enumerate(ARCHITECTURES):
        g = summary[summary.arch == arch]
        if g.empty:
            continue
        E = int(g.epochs.iloc[0])
        for i, (metric, label) in enumerate(metrics):
            ax = axes[i, j]
            ax.axhline(0, color='black', lw=0.8)
            for s in STRATEGIES:
                if s == 'none':
                    continue
                r = paired_diff(g, metric, s, 'none')
                x = n_active_epochs(s, E) / E                  # fraction du budget avec dropout
                ax.errorbar(x, r['mean'], yerr=[[r['mean'] - r['lo']], [r['hi'] - r['mean']]],
                            fmt='o', color=col(s), capsize=4, label=s)
            ax.set_title(f'{arch.upper()} ({E} époques)')
            ax.set_xlabel('fraction du budget avec dropout')
            ax.set_ylabel(label)
            ax.grid(alpha=0.3)
    axes[0, 0].legend(fontsize=8)
    finish(fig, 'duration_vs_effect')

if main_runs:
    plot_duration(main)

### 2e. Le mécanisme : les métriques de gradient de l'article

On teste maintenant l'explication de l'article, pas seulement son résultat. Trois contrastes appariés par graine, à chaque époque de mesure et pour chaque architecture (voir section 4) :

1. **effet direct des masques** : `none` mesuré avec masques contre `none` mesuré sans masque (mêmes poids) ;
2. **comparaison de l'article** : `early_30` mesuré avec masques contre `none` mesuré sans masque ;
3. **effet de trajectoire** : `early_30` contre `none`, tous deux mesurés sans masque.

Pour les trois mesures : `grad_norm` (norme du gradient), `gdv` (variance de direction), `gde` (erreur de direction).

**Prédictions de l'article à confronter** :

- les masques réduisent la norme du gradient (pas plus petits) ;
- GDV et GDE sont **plus bas** avec masques au début de l'entraînement (jusqu'à environ 3 époques chez nous). Plus tard, GDE devrait remonter : c'est le « point de bascule » ;
- la distance au point initial (panneau précédent) est plus grande avec early dropout.

**Points de vigilance.** Une réduction de GDV ou GDE n'implique pas que cette réduction *cause* le gain d'ajustement : c'est une association descriptive. Avec 5 graines et des mesures sur images propres, des écarts modestes ne sont pas fiables. Les contrastes sont exploratoires.

In [ ]:
GRAD_CONTRASTS = [   # (nom, condition A, condition B) : a - b, avec condition = stratégie|mode
    ('effet direct des masques (none)', 'none|masked', 'none|eval'),
    ('comparaison de l article (early masqué vs none)', f'{PRIMARY}|masked', 'none|eval'),
    ('effet de trajectoire (sans masque)', f'{PRIMARY}|eval', 'none|eval')]
COND_STYLE = {'none|eval': ('#555555', '-'), 'none|masked': ('#555555', '--'),
              f'{PRIMARY}|eval': ('#0072B2', '-'), f'{PRIMARY}|masked': ('#0072B2', '--')}

grad_rows = [dict(arch=r['arch'], strategy=r['strategy'], seed=r['seed'], **g)
             for r in main_runs for g in r['gradients']]
if grad_rows:
    grad_df = pd.DataFrame(grad_rows)
    grad_df['cond'] = grad_df.strategy + '|' + grad_df['mode']
    grad_df.to_csv(OUT / 'gradient_diagnostics.csv', index=False)

    # Contrastes appariés par graine, pour chaque (architecture, époque, métrique)
    rows = []
    for (arch, epoch), g in grad_df.groupby(['arch', 'epoch']):
        if epoch == 0:
            continue                      # à l'initialisation, early et none ont exactement les mêmes poids
        for metric in ('grad_norm', 'gdv', 'gde'):
            for name, a, b in GRAD_CONTRASTS:
                rows.append(dict(arch=arch, epoch=epoch, metrique=metric, contraste=name,
                                 **paired_diff(g, metric, a, b, cond='cond')))
    grad_cmp = pd.DataFrame(rows)
    grad_cmp.to_csv(OUT / 'gradient_contrasts.csv', index=False)
    display(grad_cmp.round(4))

    # Courbes : les quatre conditions, pour chaque architecture
    for arch in ARCHITECTURES:
        g = grad_df[grad_df.arch == arch]
        if g.empty:
            continue
        E = int(hist_df[hist_df.arch == arch].epoch.max())
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        for ax, (metric, title) in zip(axes, (('grad_norm', 'Norme du gradient'),
                                              ('gdv', 'GDV : variance de direction'),
                                              ('gde', 'GDE : erreur de direction'))):
            plot_band(ax, curve_ci(g, metric, 'cond'), list(COND_STYLE), style=COND_STYLE)
            ax.axvline(n_active_epochs(PRIMARY, E), color=col(PRIMARY), ls=':', alpha=0.7)
            ax.set_title(f'{arch.upper()} : {title}')
            ax.set_xlabel('époque')
            ax.grid(alpha=0.3)
        axes[0].legend(fontsize=8)
        finish(fig, f'gradients_{arch}')
else:
    print('Pas de diagnostics de gradient : lancer l étape 2 (RUN_MAIN).')

# Étape 3 : test final

**Principe.** Le test officiel (10 000 images) n'a servi à **rien** jusqu'ici : ni à choisir le régime, ni la durée, ni quoi que ce soit. On l'évalue **une seule fois**, pour **toutes** les stratégies (`none`, `standard`, `early_10`, `early_30`, `early_50`, `late_30`), toutes les architectures et toutes les graines de l'étape 2. On évalue toutes les stratégies, et pas seulement quelques-unes, parce que répondre à la question sur la durée suppose de comparer toutes les durées sur le même jeu.

**Garde-fous.**

- La cellule exige que les 90 entraînements (3 architectures × 6 stratégies × 5 graines) soient terminés.
- Un **manifeste** (empreinte SHA-256 de chaque checkpoint, régime, configuration) est écrit avec les scores. Si l'on relance avec d'autres checkpoints, la cellule refuse d'écraser des scores existants. Cela empêche de « réessayer » jusqu'à obtenir un résultat qui plaît.

**Ce que le test apporte en plus.** La consigne définit le régime de sous-apprentissage par l'accuracy de **test** : on vérifie ici que `standard − none` y est bien négatif, comme sur la validation.

**Lecture.** Mêmes outils qu'avant : différences appariées par graine, intervalles de Student, verdict, comparaisons exploratoires et non corrigées de la multiplicité.

In [ ]:
if RUN_TEST:
    regime = load_regime()
    runs = keep_regime(load_runs(OUT / 'main_runs'), regime)
    expected = len(ARCHITECTURES) * len(STRATEGIES) * len(MAIN_SEEDS)
    assert len(runs) == expected, f'{len(runs)} entraînements trouvés, {expected} attendus : terminer RUN_MAIN.'

    # Manifeste : régime + configuration + empreinte de chaque checkpoint
    manifest = json.dumps({
        'regime': regime, 'config': asdict(BASE), 'strategies': STRATEGIES,
        'checkpoints': {Path(r['checkpoint']).name: hashlib.sha256(Path(r['checkpoint']).read_bytes()).hexdigest()
                        for r in runs}}, sort_keys=True)
    csv_path, manifest_path = OUT / 'test_scores.csv', OUT / 'test_manifest.json'

    if csv_path.exists():
        # Garde-fou : on ne rejoue pas le test avec d'autres checkpoints ou un autre protocole.
        if not manifest_path.exists() or manifest_path.read_text(encoding='utf-8') != manifest:
            raise ValueError('Des scores de test existent déjà pour un autre protocole.')
        test_df = pd.read_csv(csv_path)
    else:
        rows = []
        for r in tqdm(runs):
            cfg = replace(BASE, width_mult=r['cfg']['width_mult'], epochs=r['cfg']['epochs'])
            model = build_model(r['arch'], cfg).to(device)
            model.load_state_dict(torch.load(r['checkpoint'], map_location='cpu', weights_only=True))
            score = evaluate(model, Xte, yte)
            rows.append(dict(arch=r['arch'], strategy=r['strategy'], seed=r['seed'],
                             test_loss=score['loss'], test_acc=100 * score['accuracy']))
            del model
        test_df = pd.DataFrame(rows)
        test_df.to_csv(csv_path, index=False)
        manifest_path.write_text(manifest, encoding='utf-8')

    print('Scores de test (moyenne et écart-type sur les graines) :')
    display(test_df.groupby(['arch', 'strategy'])[['test_acc', 'test_loss']].agg(['mean', 'std']).round(3))
    test_cmp = comparison_table(test_df, [('test_acc', True), ('test_loss', False)])
    test_cmp.to_csv(OUT / 'test_paired.csv', index=False)
    print('Comparaisons appariées sur le test (exploratoires, sans correction de multiplicité) :')
    display(test_cmp.round(4))
else:
    print('Test verrouillé (RUN_TEST = False). À lancer une seule fois, quand tout le reste est terminé.')

# Interprétation



## Question 1 : l'early dropout marche-t-il à notre échelle, et sur quels modèles ?


## Question 2 : y a-t-il une durée optimale ?


